# Lesson 42 · 🎨 The FFT drawn — watch the FFT's heartbeat: the butterfly diagram and the spectral fingerprints of a pure tone, a chord and noise

**Goal**: use pictures to lock the skeleton of the FFT into your head: what the butterfly network looks like, and what spectral fingerprint a pure tone, a chord and noise each leave behind.

> **🎨 How to read this lesson**: light on text, **reading the pictures comes first**; it consolidates L37–L41 and starts no long new derivations.

🔗 Aurora connection: `aurora.audio.transforms.fft` · `aurora.audio.io.sine`

← **Previous lesson** · [L41 · The windowed-FFT pipeline](L41_fft_full.ipynb)

> Last lesson covered **the windowed-FFT pipeline**: signal → window → FFT → magnitude spectrum, run end to end.
> This lesson explores **the FFT drawn**.

## Key intuition

At heart the DFT is one matrix-vector multiplication: it projects the time-domain signal onto N complex-exponential basis vectors of different frequencies. The FFT's magic is that this N×N matrix is highly redundant — fold it in half recursively and the cost drops from `O(N²)` to `O(N log N)`. Each level of folding corresponds to one column of butterfly operations in the diagram.

## Getting ready: complex exponentials and Euler's formula

Before talking about the FFT, you need to be comfortable with the idea of a "complex exponential". You'll see `exp(-2πi·k·n/N)` all over the place — what exactly is it?

**Euler's formula** is the most important bridge between complex numbers and trigonometry. It says:

$$e^{ix} = \cos(x) + i \sin(x)$$

Which means:
- at $x = \pi$, $e^{i\pi} = \cos(\pi) + i\sin(\pi) = -1$ (the most elegant equation there is)
- at $x = 2\pi$, $e^{i·2\pi} = 1$ (back to the start)
- at $x = \pi/2$, $e^{i\pi/2} = i$ (purely imaginary)

**What the complex exponential $e^{i\theta}$ looks like in the complex plane**: a **rotation**. It circles the origin, its magnitude (distance from the origin) is always 1, and its phase (angle) is $\theta$.

*Why is this useful for the DFT?*
- Frequency analysis means "sweeping" complex exponentials of different frequencies across the signal as they rotate
- If the signal happens to contain that frequency, the complex exponential lines up with it and the running sum is large (a spike)
- If it doesn't, the rotation doesn't line up and the sum stays small

**Numerical examples**:
- $e^{i·0} = 1$ (no rotation)
- $e^{i·\pi/4} = \frac{1}{\sqrt{2}} + i\frac{1}{\sqrt{2}} \approx 0.707 + 0.707i$ (rotated 45°)
- $e^{i·\pi/2} = 0 + i = i$ (rotated 90°)
- $e^{i·\pi} = -1 + 0i$ (rotated 180°)

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import sine
from aurora.audio.transforms import fft

## Key concepts: basis, projection and the conjugate

**What is a "basis"?** Picture the three mutually perpendicular standard axes of 3-D space: x, y, z. Any 3-D vector can be written as a linear combination of these three basis vectors. In the same way, the DFT uses N complex exponentials of different frequencies as its "basis vectors", and any signal of length N can be written as a weighted sum of these N basis vectors.

**What is a "projection"?** When you project a vector onto some direction, you're asking: "how much of this vector points that way?". In the DFT, the projection is $X[k]$ — the "strength" of the signal in the direction of frequency k. If $|X[k]|$ is large, that frequency component is strong in the signal.

**The complex conjugate**: for a complex number $z = a + bi$, its conjugate, written $\overline{z}$ or `conj(z)`, is defined as $\overline{z} = a - bi$. In other words, keep the real part and flip the sign of the imaginary part. For example:
- $\overline{3 + 4i} = 3 - 4i$
- $\overline{e^{i\theta}} = e^{-i\theta}$

Geometrically, the conjugate is the **mirror image across the real axis** in the complex plane.

---

## 1. The DFT matrix: an N×N unitary matrix (up to a scale factor)

**What is the DFT?** The discrete Fourier transform turns N time-domain samples into N complex frequency-domain coefficients. Mathematically:

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i \cdot k \cdot n / N}, \quad k = 0, 1, \ldots, N-1$$

This can be written as a matrix multiplication. Let $W = e^{-2\pi i/N}$ (an N-th root of unity); the DFT matrix is:

$$F[k, n] = W^{k \cdot n}, \quad k, n = 0, 1, \ldots, N-1$$

and then $\mathbf{X} = F \cdot \mathbf{x}$.

**What each row means geometrically**: row k is $[W^{k \cdot 0}, W^{k \cdot 1}, W^{k \cdot 2}, \ldots, W^{k \cdot (N-1)}]$. When you multiply this row by the signal and sum, you're taking the inner product (projection) of the signal with "the complex exponential of frequency k". The real part measures similarity to a cosine wave, the imaginary part similarity to a sine wave.

**What is a unitary matrix?** In the real case, an "orthogonal matrix" satisfies $A^T A = I$ (its transpose times itself is the identity). In the complex case we use the conjugate transpose instead (transpose, then conjugate every entry), which gives the definition of a "unitary matrix". For the DFT matrix:

$$F \cdot F^{\dagger} = N \cdot I$$

where $F^{\dagger} = \overline{F^T}$ is the conjugate transpose of F and $I$ is the identity matrix. Why the N? Because the standard DFT definition leaves out the normalization factor ($F/\sqrt{N}$ is exactly unitary). This property is crucial — it says that **the DFT preserves energy** (up to a constant factor), which is the mathematical root of Parseval's theorem below.

In the heatmaps, rows are frequency bins and columns are time samples; color encodes the real part (cosine) on the left and the imaginary part (sine) on the right.

In [ ]:
N = 16
n = np.arange(N)
k = np.arange(N)
# DFT matrix, shape (N, N)
W = np.exp(-2j * np.pi * np.outer(k, n) / N)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(W.real, cmap='RdBu_r', aspect='auto',
                     vmin=-1, vmax=1)
axes[0].set_title(f'DFT matrix, real part (cosine)  N={N}')
axes[0].set_xlabel('Time sample n')
axes[0].set_ylabel('Frequency bin k')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(W.imag, cmap='RdBu_r', aspect='auto',
                     vmin=-1, vmax=1)
axes[1].set_title(f'DFT matrix, imaginary part (sine)  N={N}')
axes[1].set_xlabel('Time sample n')
axes[1].set_ylabel('Frequency bin k')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# check unitarity: F @ conj(F).T = N * I
err = np.max(np.abs(W @ W.conj().T - N * np.eye(N)))
print(f'Unitarity error max|F·F†  - N·I| = {err:.2e}  ✅' if err < 1e-10
      else f'Unitarity error {err:.2e}  ❌')

In [ ]:
"""
Why is the real part of each DFT-matrix row a cosine and the imaginary part a sine? Check Euler's formula by hand.
"""
N = 8
k_demo = 3  # look at the basis vector for frequency 3

# by Euler's formula, the complex exponential splits into a real part (cosine) + an imaginary part (sine)
n_vals = np.arange(N)
angles = -2 * np.pi * k_demo * n_vals / N  # the angle argument

# compute directly with Euler's formula
complex_exp = np.exp(1j * angles)
real_part = np.cos(angles)
imag_part = np.sin(angles)

# check that the split is right
print(f"DFT matrix entries in row {k_demo}:")
print(f"{'n':<4} {'complex exp (re+im i)':<35} {'real (cos)':<15} {'imag (sin)':<15}")
print("-" * 70)
for n in n_vals:
    ce = complex_exp[n]
    print(f"{n:<4} {ce:<35} {real_part[n]:<15.4f} {imag_part[n]:<15.4f}")

# numerical check: the real part of the complex exponential should equal the cosine, the imaginary part the sine
err_real = np.max(np.abs(complex_exp.real - real_part))
err_imag = np.max(np.abs(complex_exp.imag - imag_part))
print(f"\n✅ Check: max|Re(exp) - cos| = {err_real:.2e}, max|Im(exp) - sin| = {err_imag:.2e}")
print("\nThat's why the real-part heatmap of the DFT matrix shows cosines and the imaginary-part heatmap shows (negative-angle) sines.")

## 2. The butterfly network: divide and conquer

As derived in section 2.5 just below, the Cooley-Tukey FFT splits a length-N DFT recursively into two DFTs of length N/2:

$$X[k]     = E[k] + W^k \cdot O[k]$$
$$X[k+N/2] = E[k] - W^k \cdot O[k]$$

where $E[k]$ is the N/2-point DFT of the even-indexed inputs, $O[k]$ that of the odd-indexed inputs, and $W^k = e^{-2\pi i k/N}$ is called the **twiddle factor**.

**What the twiddle factor means geometrically**: $W^k$ is a rotation in the complex plane. At $k=0$, $W^0 = 1$ (no rotation); at $k=1$, $W^1 = e^{-2\pi i/N}$ (a rotation by $-2\pi/N$ radians, i.e. clockwise). Every $W^k$ has magnitude $|W^k| = 1$, so it only changes the signal's **phase**, never its magnitude. In the butterfly diagram, the twiddle factor sets the phase change along the upper and lower paths of each butterfly.

For N=8 there are $\log_2(8) = 3$ stages with 4 butterflies each; the operation count is about $8 \times \log_2(8) = 24$ (vs $8^2=64$ for the naive DFT).
The diagram below draws the data flow as nodes and wires, with color marking the stage.

## 2.5 Deriving the Cooley-Tukey recursion

Now for the math behind "divide and conquer". Why does splitting an N-point DFT into two N/2-point DFTs work?

**Start from the definition**: $X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i k n / N}$

Assume N is even and split the sum by even and odd indices:

$$X[k] = \sum_{n \text{ even}} x[n] \cdot e^{-2\pi i k n / N} + \sum_{n \text{ odd}} x[n] \cdot e^{-2\pi i k n / N}$$

Let $n = 2m$ (even) or $n = 2m+1$ (odd), with $m = 0, 1, \ldots, N/2-1$:

$$X[k] = \sum_{m=0}^{N/2-1} x[2m] \cdot e^{-2\pi i k (2m) / N} + \sum_{m=0}^{N/2-1} x[2m+1] \cdot e^{-2\pi i k (2m+1) / N}$$

Split the exponent in the second term:

$$= \sum_{m=0}^{N/2-1} x[2m] \cdot e^{-2\pi i k m / (N/2)} + e^{-2\pi i k / N} \sum_{m=0}^{N/2-1} x[2m+1] \cdot e^{-2\pi i k m / (N/2)}$$

Let $E[k] = \sum_{m=0}^{N/2-1} x[2m] \cdot e^{-2\pi i k m / (N/2)}$ (the N/2-point DFT of the even-indexed samples)

Let $O[k] = \sum_{m=0}^{N/2-1} x[2m+1] \cdot e^{-2\pi i k m / (N/2)}$ (the N/2-point DFT of the odd-indexed samples)

Let $W^k = e^{-2\pi i k / N}$ (the twiddle factor). Then:

$$X[k] = E[k] + W^k \cdot O[k]$$

This holds for $k = 0, 1, \ldots, N/2-1$.

**What about $k = N/2, N/2+1, \ldots, N-1$?** Use periodicity. $E[k]$ and $O[k]$ are both N/2-point DFTs, so they repeat every N/2:

$$E[k + N/2] = E[k], \quad O[k + N/2] = O[k]$$

and $W^{k+N/2} = e^{-2\pi i (k+N/2) / N} = e^{-2\pi i k / N} \cdot e^{-i\pi} = -W^k$

So for $k = 0, 1, \ldots, N/2-1$:

$$X[k + N/2] = E[k] - W^k \cdot O[k]$$

**A small worked example (N=4)**: input $x = [1, 2, 3, 4]$

```
even part: x[0]=1, x[2]=3  → E = FFT([1,3]) (2-point)
odd part:  x[1]=2, x[3]=4  → O = FFT([2,4]) (2-point)
W^0 = 1, W^1 = e^(-iπ/2) = -i
E = [1+3, 1-3] = [4, -2]
O = [2+4, 2-4] = [6, -2]
X[0] = E[0] + W^0·O[0] = 4 + 6       = 10
X[1] = E[1] + W^1·O[1] = -2 + (-i)(-2) = -2+2i
X[2] = E[0] - W^0·O[0] = 4 - 6       = -2
X[3] = E[1] - W^1·O[1] = -2 - (-i)(-2) = -2-2i
```

This recursion is where the "butterfly" comes from.


In [ ]:
# Draw the N=8 butterfly network (3 stages, 4 butterflies each)
N = 8
stages = int(np.log2(N))  # 3

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(-0.3, stages + 0.3)
ax.set_ylim(-0.5, N - 0.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('N=8 butterfly network (3 stages × 4 butterflies)', fontsize=13)

# bit-reversal permutation order (input order)
bit_rev = [int(f'{i:03b}'[::-1], 2) for i in range(N)]

# node y coordinates (input side arranged in bit-reversed order)
y_pos = {i: bit_rev[i] for i in range(N)}
# spread evenly from stage 0 to stage 3
colors = plt.cm.plasma(np.linspace(0.15, 0.85, stages))

def node_xy(stage, wire):
    return stage, wire

# draw the input labels (in bit-reversed order)
for i in range(N):
    ax.text(-0.15, bit_rev[i], f'x[{i}]', va='center', ha='right', fontsize=8)

# draw the butterflies stage by stage
# current_y = list(range(N))  # reserved: track each wire's y coordinate (not used yet)
for s in range(stages):
    half = 2 ** s          # butterfly span
    group = 2 * half       # group size
    n_butterflies = N // group  # number of butterfly groups in this stage
    col_in  = s
    col_out = s + 1
    for g in range(n_butterflies):
        for b in range(half):
            top = g * group + b
            bot = top + half
            yt, yb = top, bot
            # twiddle-factor phase (0..0.5 mapped to opacity, to show the phase change)
            phase = b / group  # 0..0.5
            alpha_cross = 0.4 + 0.5 * phase  # the larger the phase, the more visible the diagonal
            c = colors[s]
            # straight-through wire (top)
            ax.annotate('', xy=(col_out, yt), xytext=(col_in, yt),
                        arrowprops=dict(arrowstyle='->', color=c, lw=1.4))
            # straight-through wire (bottom)
            ax.annotate('', xy=(col_out, yb), xytext=(col_in, yb),
                        arrowprops=dict(arrowstyle='->', color=c, lw=1.4))
            # cross wire top→bot (the +W^k branch)
            ax.plot([col_in, col_out], [yt, yb], color=c, lw=0.8, ls='--', alpha=alpha_cross)
            # cross wire bot→top (the -W^k branch, completing the butterfly)
            ax.plot([col_in, col_out], [yb, yt], color=c, lw=0.8, ls=':', alpha=alpha_cross)
            # twiddle-factor label (on every butterfly; exponent = b * N // group)
            mid_x = (col_in + col_out) / 2
            mid_y = (yt + yb) / 2
            exp_k = b * N // group  # the correct twiddle-factor exponent
            ax.text(mid_x, mid_y + 0.15,
                    f'W^({N})^{{{exp_k}}}', fontsize=5.5, color=c,
                    ha='center', va='bottom')

# output labels
for i in range(N):
    ax.text(stages + 0.15, i, f'X[{i}]', va='center', ha='left', fontsize=8)

# stage labels
for s in range(stages):
    ax.text(s + 0.5, N - 0.1, f'Stage {s+1} ', ha='center', fontsize=9,
            color=colors[s], fontweight='bold')

plt.tight_layout()
plt.show()
print(f'N={N}: {stages} stages, {N//2} butterflies per stage, {N//2 * stages} butterflies in total (vs N² = {N*N} multiply-adds for the naive DFT)')
print('Butterfly: X[top] = E + W^k·O (-- wire), X[bot] = E − W^k·O (: wire)')


In [ ]:
"""
Demonstrate the bit-reversal permutation
"""
N = 8
log_N = 3  # log2(8)

# compute the bit-reversal mapping
bit_rev = []
for i in range(N):
    # turn i into a binary string (3 bits, zero-padded), reverse it, convert back to decimal
    binary_str = f'{i:0{log_N}b}'
    reversed_str = binary_str[::-1]
    reversed_idx = int(reversed_str, 2)
    bit_rev.append(reversed_idx)

print("Bit-reversal mapping:")
print(f"{'index i':<8} {'binary':<10} {'reversed':<10} {'new idx':<8}")
print("-" * 36)
for i in range(N):
    binary_str = f'{i:0{log_N}b}'
    reversed_str = binary_str[::-1]
    print(f"{i:<8} {binary_str:<10} {reversed_str:<10} {bit_rev[i]:<8}")

# suppose the input signal is [0, 1, 2, 3, 4, 5, 6, 7]
x_original = np.arange(N, dtype=float)
x_reordered = np.array([x_original[bit_rev[i]] for i in range(N)])

print(f"\nOriginal order: {x_original}")
print(f"Reordered:      {x_reordered}")
print(f"Rule: at new positions {bit_rev} goes the original x[index] (new position i holds x[bit_rev[i]])")


## 2.1 The bit-reversal permutation

Why does an iterative FFT have to reorder its input in bit-reversed order? It comes from the **bottom-up** order in which the butterfly network is computed.

**The idea**: the naive DFT is a single matrix multiplication done in one go. The Cooley-Tukey recursion, though, computes in three stages of butterflies, and each stage reads the previous stage's output. To avoid shuffling data back and forth (which would make memory access very inefficient), we reorder once at the **input**, and from then on every stage of butterflies can read and write in order.

The bit-reversal rule is simple: reverse the binary digits of index i to get its new position. For example, with N=8 (3 binary digits):

| Index i | Binary | Reversed | New position |
|---------|--------|--------|--------|
| 0 | 000 | 000 | 0 |
| 1 | 001 | 100 | 4 |
| 2 | 010 | 010 | 2 |
| 3 | 011 | 110 | 6 |
| 4 | 100 | 001 | 1 |
| 5 | 101 | 101 | 5 |
| 6 | 110 | 011 | 3 |
| 7 | 111 | 111 | 7 |

So $[x[0], x[1], x[2], x[3], x[4], x[5], x[6], x[7]]$ becomes $[x[0], x[4], x[2], x[6], x[1], x[5], x[3], x[7]]$ after reordering.

**Key observation**: after reordering, neighbors are no longer the original neighbors; instead, elements that are "neighbors in binary" end up next to each other. That way the first stage of butterflies (span 1) handles these pairs, the second stage (span 2) handles larger groups… and so on, with every stage reading contiguous blocks of data.

The code cell just above walks through bit reversal for N=8.

## 3. Spectral symmetry: the mirror structure of a real signal

When the input `x[n]` is **real**, the DFT is conjugate-symmetric:

```
X[N-k] = conj(X[k]),   k = 1, 2, ..., N/2-1
```

So the magnitude spectrum `|X[k]|` is symmetric about `k = N/2`, and only the first half (`k = 0..N/2`) carries independent information.
In the plot you'll see the left and right halves of the magnitude mirror each other, while the phase flips sign.
Aurora's `fft` returns all N bins; when you use it, just take `[:N//2+1]`.

In [ ]:
"""
Compute an N=4 DFT by hand and check conjugate symmetry
"""
N = 4
x = np.array([1, 2, 3, 4], dtype=float)

# compute each bin by hand
print("Computing the four components of an N=4 DFT by hand:\n")
X_manual = []
for k in range(N):
    sum_val = 0 + 0j
    print(f"X[{k}] = sum_n x[n] * exp(-2πi*{k}*n/4)")
    for n in range(N):
        angle = -2 * np.pi * k * n / N
        exp_val = np.exp(1j * angle)
        term = x[n] * exp_val
        sum_val += term
        print(f"  n={n}: x[{n}]={x[n]} * exp({angle/np.pi:.2f}π·i) = {x[n]} * ({exp_val.real:.3f}+{exp_val.imag:.3f}i) = {term}")
    X_manual.append(sum_val)
    print(f"  → X[{k}] = {sum_val.real:.3f} + {sum_val.imag:.3f}i\n")

X_manual = np.array(X_manual)

# check against numpy's FFT
X_numpy = np.fft.fft(x)

# check conjugate symmetry: X[N-k] = conj(X[k])
print("\nChecking conjugate symmetry X[N-k] = conj(X[k]):")
print(f"{'k':<3} {'X[k]':<25} {'conj(X[k])':<25} {'X[N-k]':<25} {'error':<15}")
print("-" * 85)
for k in range(N):
    conj_Xk = np.conj(X_manual[k])
    X_Nmk = X_manual[N - k] if N - k < N else X_manual[N - k - N]  # handle the wrap-around
    err = np.abs(X_Nmk - conj_Xk)
    print(f"{k:<3} {str(X_manual[k]):<25} {str(conj_Xk):<25} {str(X_Nmk):<25} {err:.2e}")

# numerical check
print(f"\n✅ Max conjugate-symmetry error = {np.max(np.abs(X_manual[1:N//2] - np.conj(X_manual[N-1:N//2:-1]))):.2e}")


## 3.5 Why must the DFT of a real signal be conjugate-symmetric?

It's no coincidence — it follows straight from the definition.

**Assumption**: the input $x[n]$ is real (no imaginary part).

**Definition**: the k-th DFT component is
$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i kn/N}$$

Now look at component $N-k$:
$$X[N-k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i (N-k)n/N}$$

$$= \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i n} \cdot e^{2\pi i kn/N}$$

Because $e^{-2\pi i n} = 1$ (a whole number of turns around the circle):

$$= \sum_{n=0}^{N-1} x[n] \cdot e^{2\pi i kn/N}$$

By Euler's formula $e^{i\theta} = \cos(\theta) + i\sin(\theta)$, conjugating just flips the sign of the imaginary part:

$$\overline{e^{i\theta}} = e^{-i\theta}$$

So:
$$X[N-k] = \sum_{n=0}^{N-1} x[n] \cdot \overline{e^{-2\pi i kn/N}} = \overline{\sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i kn/N}} = \overline{X[k]}$$

That is, **$X[N-k] = \overline{X[k]}$** (conjugate symmetry). This is the fundamental property of the DFT of a real signal.

**Geometric meaning**: in the frequency domain, the positive- and negative-frequency components are mirror images (across the real axis). A real signal carries no independent information at negative frequencies, so its magnitude spectrum is symmetric about $k=N/2$.

**Worked example by hand (N=4)**: input $x = [1, 2, 3, 4]$ (real) — see the code cell above.


In [ ]:
# Demonstrate conjugate symmetry with aurora.audio.transforms.fft
sr = 8000
duration = 0.016  # 16 ms → N=128 samples (2^7; the aurora FFT needs a power of 2)
x = sine(440.0, sample_rate=sr, duration=duration)  # real signal
N = len(x)

X = fft(x)          # aurora FFT, returns a complex spectrum
mag   = np.abs(X)
phase = np.angle(X)

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
bins = np.arange(N)

axes[0].stem(bins, mag, markerfmt='C0o', linefmt='C0-', basefmt='k-')
axes[0].axvline(N//2, color='red', ls='--', lw=1, label=f'k=N/2={N//2}')
axes[0].set_ylabel('Magnitude |X[k]|')
axes[0].set_title(f'Magnitude spectrum of a real 440 Hz signal (N={N}) — symmetric about k=N/2')
axes[0].legend()

axes[1].stem(bins, phase, markerfmt='C1o', linefmt='C1-', basefmt='k-')
axes[1].axvline(N//2, color='red', ls='--', lw=1)
axes[1].set_ylabel('Phase ∠X[k]  (rad)')
axes[1].set_xlabel('Frequency bin k')
axes[1].set_title('Phase spectrum — the right half mirrors the left half with the sign flipped')

plt.tight_layout()
plt.show()

# numerical check: X[N-k] = conj(X[k])
err = np.max(np.abs(X[1:N//2] - np.conj(X[N-1:N//2:-1])))
print(f'Conjugate-symmetry error max|X[N-k] - conj(X[k])| = {err:.2e}',
      '✅' if err < 1e-10 else '❌')

## 🏋️ Exercise: implement a three-stage N=8 Cooley-Tukey FFT by hand

**Task**: without calling any `fft` function, implement an iterative Cooley-Tukey FFT of length 8 in NumPy,
and compare its output numerically with `aurora.audio.transforms.fft`.

**Step hints**:
1. Reorder the length-8 input by bit reversal (the bit-reversal permutation): `bit_rev = [int(f'{i:03b}'[::-1], 2) for i in range(8)]`
2. Stage 1 (`half=1`): butterflies on adjacent pairs: `new_top = x[top] + W^0 * x[bot]`
3. Stage 2 (`half=2`): elements 2 apart, twiddle factor `W^k = exp(-2πi·k/8)`
4. Stage 3 (`half=4`): elements 4 apart, same twiddle factor
5. Compare with `aurora.audio.transforms.fft(x)`; the error should be below `1e-10`

**Aurora connection**: `aurora.audio.transforms.fft` is the production version of exactly this algorithm. When you're done, compare it with the source and see what differs.


In [ ]:
"""
Complex multiplication in practice
"""
print("Complex multiplication example (the rotation inside a butterfly):\n")

# twiddle factor W^k
k = 1
N = 8
W_k = np.exp(-2j * np.pi * k / N)
print(f"Twiddle factor W^{k}_{N} = exp(-2πi·{k}/{N}) = {W_k}")
print(f"  real part (cosine) = {W_k.real:.6f}")
print(f"  imaginary part (sine) = {W_k.imag:.6f}")

# the complex number to rotate
X_bot = 1 + 2j
print(f"\nSignal value X[bot] = {X_bot}")

# butterfly step: multiply by the twiddle factor
result = W_k * X_bot
print(f"\nRotation: W^{k} · X[bot] = {result}")
print(f"  real part = {result.real:.6f}")
print(f"  imaginary part = {result.imag:.6f}")

# check by hand: the complex multiplication formula
a, b = W_k.real, W_k.imag  # W_k = a + bi
c, d = X_bot.real, X_bot.imag  # X = c + di
manual_real = a*c - b*d
manual_imag = a*d + b*c
print(f"\nChecking by hand (formula (a+bi)(c+di) = (ac-bd) + (ad+bc)i):")
print(f"  (a,b) = ({a:.6f}, {b:.6f})  [twiddle factor]")
print(f"  (c,d) = ({c:.6f}, {d:.6f})  [signal value]")
print(f"  real = {a:.6f}·{c:.6f} - {b:.6f}·{d:.6f} = {manual_real:.6f} ✓")
print(f"  imag = {a:.6f}·{d:.6f} + {b:.6f}·{c:.6f} = {manual_imag:.6f} ✓")

print(f"\nTakeaway: NumPy's complex multiplication handles these details for you; no need to do it by hand.")

In [ ]:
"""
Demonstrate the index logic of the butterfly loops
"""
N = 8
stages = 3

print(f"N={N}, {stages} stages of butterflies; the index pairs in each stage:\n")

for s in range(stages):
    half = 2 ** s
    group = 2 * half
    n_groups = N // group

    print(f"Stage {s+1} (stage {s}): half={half}, group={group}, n_groups={n_groups}")
    print(f"  groups and butterfly pairs:")

    pairs = []
    for g in range(n_groups):
        for b in range(half):
            top = g * group + b
            bot = top + half
            pairs.append((top, bot))
            print(f"    group {g}, butterfly {b}: (top={top}, bot={bot})")
    print()

print("The pattern:")
print("  Stage 1 (half=1): butterflies on adjacent pairs (0,1), (2,3), (4,5), (6,7)")
print("  Stage 2 (half=2): butterflies 2 apart (0,2), (1,3), (4,6), (5,7)")
print("  Stage 3 (half=4): butterflies 4 apart (0,4), (1,5), (2,6), (3,7)")
print("\nThe butterfly span doubles each stage, so every pair of data gets fully combined.")

## Toolbox before the exercise: complex multiplication and the butterfly loop structure

Before you start coding, there are two things to understand.

### How is complex multiplication computed?

In DFT butterflies you'll often see `X[bot] = X[top] - W_k * X[bot]`, where `W_k` is complex and so is `X[bot]`. The rule for complex multiplication:

$$(a + bi) \times (c + di) = (ac - bd) + (ad + bc)i$$

NumPy already takes care of this; you just write `W_k * X_bot` and it applies the formula above automatically.

**Numerical example**:
- $W_k = e^{-i\pi/4} = \cos(\pi/4) - i\sin(\pi/4) = 0.707 - 0.707i$
- $X = 1 + 2i$
- $W_k \times X = (0.707 - 0.707i) \times (1 + 2i)$
  $= 0.707 \times 1 + 0.707 \times 2i - 0.707i \times 1 - 0.707i \times 2i$
  $= 0.707 + 1.414i - 0.707i + 1.414$ (because $i^2 = -1$)
  $= 2.121 + 0.707i$

### The index logic of the butterfly loops

What do the four variables in the three nested loops mean?

```python
for s in range(3):          # s = stage, 0, 1, 2
    half = 2**s             # half = 1, 2, 4 (butterfly span)
    group = 2 * half        # group = 2, 4, 8 (size of each group)
    n_butterflies = N // group  # how many "groups" in this stage

    for g in range(n_butterflies):  # g = group index (which group)
        for b in range(half):       # b = butterfly index in group (which butterfly within the group)
            top = g * group + b         # upper endpoint
            bot = top + half            # lower endpoint
```

**Intuition**:
- Stage 1 (half=1): the 8 elements form 4 groups (group=2) of 2; butterflies on neighbors
- Stage 2 (half=2): the 8 elements form 2 groups (group=4) of 4; butterflies 2 apart
- Stage 3 (half=4): the 8 elements form 1 group (group=8), the whole array; butterflies 4 apart

**Why `top = g * group + b, bot = top + half`?**
- `g * group + b` is element b of group g
- `top + half` is the element `half` positions further on (the two endpoints of a butterfly)

**Code check**: the two code cells just above walk through both.

## 3.8 Parseval's theorem: conservation of energy

**Question**: how should a signal's energy (sum of squares) in the time domain relate to its energy in the frequency domain? Are they necessarily equal?

**Answer**: yes, up to a scale factor. That's **Parseval's theorem**:

$$\sum_{n=0}^{N-1} |x[n]|^2 = \frac{1}{N} \sum_{k=0}^{N-1} |X[k]|^2$$

The left side is the **time-domain energy**, the right side the **frequency-domain energy** (times $1/N$).

**Why the $1/N$?** It comes from how the DFT is normalized. Different DFT definitions carry different factors:
- the standard DFT: $X[k] = \sum_n x[n] e^{-2\pi i kn/N}$ (no 1/N) → Parseval needs the 1/N
- some texts use $X[k] = \frac{1}{\sqrt{N}} \sum_n x[n] e^{...}$ (the unitary normalization) → Parseval is a plain equality

**Physical meaning**: whether you look at a signal's strength in the time domain or at the contributions of different frequencies, the total energy is conserved (up to a constant factor). This reflects the DFT's **unitarity** — it neither "creates" nor "destroys" energy.

**Numerical check**: see check 3 in the property cell at the end of this lesson.

In [ ]:
def my_fft_n8(x):
    """Iterative Cooley-Tukey FFT for N=8, by hand (no FFT library calls).

    Parameters: x — a real or complex numpy array of length 8
    Returns: a complex spectrum array of length 8
    """
    N = 8
    assert len(x) == N, f"needs an N=8 input, got N={len(x)}"

    # TODO: Step 1 — bit-reversal permutation
    # bit_rev = [int(f'{i:03b}'[::-1], 2) for i in range(N)]
    # X = np.array([x[bit_rev[i]] for i in range(N)], dtype=complex)
    raise NotImplementedError("Implement the bit-reversal permutation and the butterfly iterations")

    # TODO: Step 2 — three stages of butterflies
    # for s in range(3):
    #     half = 2**s
    #     group = 2 * half
    #     for g in range(N // group):
    #         for b in range(half):
    #             top = g * group + b
    #             bot = top + half
    #             W_k = np.exp(-2j * np.pi * b / group)
    #             t = X[top] + W_k * X[bot]
    #             X[bot] = X[top] - W_k * X[bot]
    #             X[top] = t

    return X


# ── Check (wrapped in try/except so it doesn't crash before you implement it) ──
try:
    np.random.seed(42)
    x_test = np.random.randn(8)
    result_manual = my_fft_n8(x_test)
    result_aurora = fft(x_test)

    err = np.max(np.abs(result_manual - result_aurora))
    assert err < 1e-10, f"error too large: {err:.2e} (expected < 1e-10)"
    print(f"✅ Check passed: max error between your FFT and aurora.fft = {err:.2e}")
    print(f"Your result X[0..3] = {result_manual[:4]}")
except (NotImplementedError, TypeError):
    print("⚠️  Exercise not implemented yet; fill in my_fft_n8 and run this cell again")
except AssertionError as e:
    print(f"❌ Check failed: {e}")


## 3.6 Powers of 2: why does the aurora FFT require them?

**Short answer**: the Cooley-Tukey recursion folds N into two halves of N/2 every time. Only when N is a power of 2 can you keep folding all the way down to N=1 (the trivial case).

**In detail**:
- N=8=2³: 8 → 4+4 → 2+2+2+2 → 1+1+…+1 (works)
- N=6=2×3: 6 → 3+3 → stuck! 3 is odd and can't be folded again

So the standard Cooley-Tukey algorithm only works for $N = 2^m$.

**What if your signal has N=400 samples?** Use **zero-padding**:
- find the smallest m with $2^m \ge 400$, i.e. m=9 ($2^9=512$)
- pad the signal from 400 to 512 samples (112 zeros at the end)
- compute a 512-point FFT; the result has 512 bins
- the bin spacing gets finer: it was sr/400 and is now sr/512

This only changes the **bin spacing** (it samples the same underlying spectrum more densely; it doesn't add true resolution), and it doesn't move the peaks (the frequencies haven't changed).

## Parameter experiment: a two-tone signal (stacked pure tones = the seed of a chord)

**Signal parameters**: `sr=8000`, `duration=0.032` (32 ms, N=256=2⁸),
stacking `f1=440 Hz` (bin ≈ 14) and `f2=880 Hz` (bin ≈ 28).

> **Note**: the aurora FFT needs an input length that's a power of 2. Why? The Cooley-Tukey recursion halves every time, and only a power of 2 halves all the way down. N=400 isn't a power of 2, so it has to be zero-padded to the next power of 2 (512).

**What you should see**:
- two peaks in the magnitude spectrum, each with a mirror image, at bins of about `round(f * N / sr)`
- the 880 Hz bin is exactly twice the 440 Hz bin (bin 28 = 2 × bin 14, an octave)
- above `N/2=128`, the mirror peaks show up symmetrically on the high-frequency side (conjugate symmetry)

In [ ]:
"""
Check the bin-position formula: bin = round(f * N / sr)
"""
sr = 8000
N = 256
delta_f = sr / N  # frequency resolution (Hz/bin)

# a few test frequencies
test_freqs = [440, 880, 1320, 4000]

print(f"Sample rate sr = {sr} Hz")
print(f"Samples N = {N}")
print(f"Frequency resolution Δf = sr/N = {delta_f:.2f} Hz/bin")
print(f"Highest representable frequency = sr/2 = {sr/2} Hz\n")

print(f"{'freq f (Hz)':<15} {'exact bin':<15} {'rounded':<15} {'check: Hz of that bin':<25}")
print("-" * 70)

for f in test_freqs:
    exact_bin = f * N / sr
    rounded_bin = round(exact_bin)
    freq_of_bin = rounded_bin * delta_f  # reverse check: what frequency is this bin?
    print(f"{f:<15} {exact_bin:<15.2f} {rounded_bin:<15} {freq_of_bin:<25.2f}")

# check graphically: build a 440 + 880 Hz signal and look at where the FFT peaks are
print(f"\nBuilding the signal: 440 Hz + 880 Hz (N={N} samples)")
duration = N / sr  # seconds
t = np.arange(N) / sr
x = np.sin(2 * np.pi * 440 * t) + np.sin(2 * np.pi * 880 * t)

X = np.fft.fft(x)
mag = np.abs(X)

peak_bins = np.argsort(mag)[-4:][::-1]  # take the 4 largest peaks
peak_freqs = peak_bins * delta_f

print(f"{'Peak index':<15} {'Frequency (Hz)':<20} {'Expected':<20}")
for peak_bin, peak_freq in zip(peak_bins, peak_freqs):
    print(f"{peak_bin:<15} {peak_freq:<20.1f} {'' if peak_bin <= N // 2 else '(mirror)'}")


## 3.7 Deriving the bin-position formula

A lot of students see `bin = round(f * N / sr)` and wonder: "why that formula?". Let's derive it from the sampling setup.

**Background**:
- sample rate `sr` (in Hz, i.e. samples per second)
- number of samples `N`
- total duration $T = N / sr$ (seconds)

**DFT output**: N bins, covering the frequency range $[0, sr)$. Why stop at `sr`? Because of the Nyquist theorem: the highest representable frequency is $sr/2$, and the bins above it are mirror images.

**Bin spacing (frequency resolution)**: $\Delta f = sr / N$ (Hz/bin)

Which means:
- bin 0 is at $0 \times \Delta f = 0$ Hz
- bin 1 is at $1 \times \Delta f = sr/N$ Hz
- bin k is at $k \times \Delta f = k \times sr/N$ Hz
- bin N-1 is at $(N-1) \times sr/N$ Hz

**Going the other way**: if the signal has a frequency of $f$ Hz, which bin does it land in?

If bin k is at frequency $f$, then:
$$k = \frac{f}{\Delta f} = \frac{f}{sr/N} = \frac{f \cdot N}{sr}$$

Since k has to be an integer, the actual bin is $\text{round}(f \cdot N / sr)$.

**Numerical example**:
- sr = 8000 Hz, N = 256, $\Delta f = 8000/256 = 31.25$ Hz/bin
- 440 Hz lands in bin round(440 × 256 / 8000) = round(14.08) = 14
- 880 Hz lands in bin round(880 × 256 / 8000) = round(28.16) = 28

The code cell just above checks this formula.

In [ ]:
sr       = 8000
duration = 0.032   # 32 ms → N=256 (2^8)
f1, f2   = 440.0, 880.0

x1 = sine(f1, sample_rate=sr, duration=duration)
x2 = sine(f2, sample_rate=sr, duration=duration)
x  = x1 + x2
N  = len(x)

X   = fft(x)
mag = np.abs(X)

# theoretical peak bins
bin1 = round(f1 * N / sr)
bin2 = round(f2 * N / sr)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(mag, color='steelblue', lw=1.2)
ax.axvline(bin1, color='C1', ls='--', label=f'{f1:.0f} Hz  bin={bin1}')
ax.axvline(N - bin1, color='C1', ls=':', alpha=0.5, label=f'mirror bin={N-bin1}')
ax.axvline(bin2, color='C2', ls='--', label=f'{f2:.0f} Hz  bin={bin2}')
ax.axvline(N - bin2, color='C2', ls=':', alpha=0.5, label=f'mirror bin={N-bin2}')
ax.axvline(N//2, color='red', ls='-', lw=0.8, alpha=0.4, label=f'N/2={N//2}')
ax.set_xlabel('Frequency bin k')
ax.set_ylabel('Magnitude |X[k]|')
ax.set_title(f'Spectrum of a 440 Hz + 880 Hz mix  (sr={sr}, N={N})')
ax.legend()
plt.tight_layout()
plt.show()

print(f'440 Hz → bin {bin1}  (exact: {f1*N/sr:.2f})')
print(f'880 Hz → bin {bin2}  (exact: {f2*N/sr:.2f})')
print(f'Ratio of the two peak bins: {bin2}/{bin1} = {bin2/bin1:.2f}  (should be ≈2.0, an octave)')

## 4. Spectral shapes compared: pure tone / chord / white noise

The three kinds of signal promised in the title meet here — compare their spectral shapes at a glance:

| Signal | Spectral signature | Intuition |
|---|---|---|
| **Pure tone** (a single sine) | one sharp spike + its mirror | energy concentrated at one frequency |
| **Chord** (several frequencies stacked) | several discrete peaks | energy spread over a few frequencies |
| **White noise** | broadband, flat response | energy spread evenly over all frequencies |


In [ ]:
sr_cmp  = 8000
dur_cmp = 0.032  # 32 ms → N=256

# ── Pure tone: a single 440 Hz sine ──
x_pure  = sine(440.0, sample_rate=sr_cmp, duration=dur_cmp)

# ── Chord: 440 + 660 + 880 Hz stacked ──
x_chord = (sine(440.0, sample_rate=sr_cmp, duration=dur_cmp) +
           sine(660.0, sample_rate=sr_cmp, duration=dur_cmp) +
           sine(880.0, sample_rate=sr_cmp, duration=dur_cmp))

# ── White noise: np.random.randn, equal energy at all frequencies in theory ──
np.random.seed(0)
N_cmp   = len(x_pure)
x_noise = np.random.randn(N_cmp).astype(float)

# magnitude spectrum of each signal (only the first N//2+1 positive-frequency bins)
def half_spectrum(x):
    X = fft(x)
    return np.abs(X[:len(x)//2 + 1])

mag_pure  = half_spectrum(x_pure)
mag_chord = half_spectrum(x_chord)
mag_noise = half_spectrum(x_noise)

freqs = np.arange(len(mag_pure)) * sr_cmp / N_cmp

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(freqs, mag_pure, color='C0', lw=1.4)
axes[0].set_title('Pure tone (440 Hz) — a single spike', fontsize=11)
axes[0].set_ylabel('Magnitude |X[k]|')

axes[1].plot(freqs, mag_chord, color='C2', lw=1.4)
axes[1].set_title('Chord (440 + 660 + 880 Hz) — three discrete peaks', fontsize=11)
axes[1].set_ylabel('Magnitude |X[k]|')

axes[2].plot(freqs, mag_noise, color='C3', lw=1.0, alpha=0.8)
axes[2].set_title('White noise (randn) — broadband and flat (with random ripples)', fontsize=11)
axes[2].set_ylabel('Magnitude |X[k]|')
axes[2].set_xlabel('Frequency (Hz)')

for ax in axes:
    ax.set_xlim(0, sr_cmp / 2)

plt.suptitle('Spectral shapes compared: pure tone / chord / white noise', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# numerical check: spread of the noise spectrum relative to its mean (a rough flatness check)
noise_std_ratio = mag_noise.std() / mag_noise.mean()
print(f'Pure tone   peak bin of the magnitude spectrum = {np.argmax(mag_pure)} ({freqs[np.argmax(mag_pure)]:.0f} Hz)')
print(f'Chord       top 3 peak bins = {np.argsort(mag_chord)[-3:][::-1]} → Hz ≈ {freqs[np.argsort(mag_chord)[-3:][::-1]].astype(int)}')
print(f'White noise coefficient of variation of the magnitude = {noise_std_ratio:.2f} (smaller = flatter; for white noise theory gives ≈0.52)')


## Wrap-up

`fft` (from `aurora.audio.transforms`) maps an N-point real signal to N complex bins.
Visualizing the DFT matrix shows you that every row is a complex-exponential basis vector; the butterfly network lays the recursive `O(N log N)` folding structure out in front of you; and conjugate symmetry means a real signal only needs its first `N//2+1` bins.
The next lesson, `L43_stft.ipynb`, hooks the FFT up to a sliding window, so you can finally watch frequencies evolve over time.

In [ ]:
# ── Standalone math assertions: checking the core properties of the FFT (nothing for you to implement) ──
import numpy as np

np.random.seed(7)
x = np.random.randn(16)
N = len(x)
X = np.fft.fft(x)

# 1. Unitarity of the DFT matrix: F @ conj(F).T = N * I
n_arr = np.arange(N)
F = np.exp(-2j * np.pi * np.outer(n_arr, n_arr) / N)
err_unitary = np.max(np.abs(F @ F.conj().T - N * np.eye(N)))
assert err_unitary < 1e-10, f"unitarity error too large: {err_unitary:.2e}"
print(f"1 ✅  DFT matrix unitarity: max|F·F†-N·I| = {err_unitary:.2e}")

# 2. Conjugate symmetry: X[N-k] = conj(X[k]), k=1..N/2-1
err_sym = np.max(np.abs(X[1:N//2] - np.conj(X[N-1:N//2:-1])))
assert err_sym < 1e-10, f"conjugate-symmetry error: {err_sym:.2e}"
print(f"2 ✅  Conjugate symmetry: max|X[N-k]-X[k]*| = {err_sym:.2e}")

# 3. Parseval's theorem: Σ|x|² = (1/N)·Σ|X|²
energy_time = np.sum(np.abs(x)**2)
energy_freq = np.sum(np.abs(X)**2) / N
assert np.isclose(energy_time, energy_freq, rtol=1e-10), \
    f"Parseval failed: time domain={energy_time:.6f}, frequency domain (1/N)={energy_freq:.6f}"
print(f"3 ✅  Parseval's theorem:")
print(f"     time-domain energy Σ|x[n]|² = {energy_time:.6f}")
print(f"     frequency-domain energy (1/N)Σ|X[k]|² = {energy_freq:.6f}")
print(f"     difference = {abs(energy_time - energy_freq):.2e}")

# 4. Linearity: FFT(ax+by) = a·FFT(x) + b·FFT(y)
y = np.random.randn(N); a, b = 2.5, -1.3
assert np.allclose(np.fft.fft(a*x + b*y), a*np.fft.fft(x) + b*np.fft.fft(y), atol=1e-10)
print(f"4 ✅  Linearity: FFT(ax+by) = a·FFT(x)+b·FFT(y)")

# 5. DC component: X[0] = Σ x[n]
assert np.isclose(X[0], np.sum(x), atol=1e-10)
print(f"5 ✅  DC component: X[0]={X[0]:.6f}, Σx={np.sum(x):.6f}")

print("\n🎓 These properties aren't mathematical coincidences; they follow necessarily from the DFT being a (scaled) unitary transform.")

## Appendix · From "one FFT of the whole segment" to the STFT: one extra dimension

L41 answers: **which frequencies are in this whole windowed segment?** The output is a 1-D spectrum.

The STFT answers: **which frequencies are in each slice of time?** The output is a 2-D matrix:

```text
STFT shape ≈ (n_frames, n_bins)   # time × frequency
```

Parts you already have:

| Part | From lesson | Job |
|---|---|---|
| window `w` | L36 | tames the edges of each frame |
| FFT | L39 | turns each frame into a spectrum |
| framing | L43 (next lesson) | slides along and cuts out `(n_frames, win_len)` |

Three new parameters:

- `win_len`: frame length; bigger means finer frequency (Δf = sr / win_len)
- `hop`: frame step; smaller means denser in time (Δt = hop / sr)
- `window`: usually Hann (L36)

Frame-count formula (without center padding):

```text
n_frames = 1 + (len(x) - win_len) // hop
```

ASCII intuition:

```text
x:  |----|----|----|----|----|----|
       [win]
          [win]   ← step by hop
             [win]
```

L44 isn't a new algorithm; it chains **framing → windowing → FFT** into `my_stft`.


---

→ **Next lesson** · [L43 · How the STFT works](L43_stft.ipynb)

> Next lesson covers **how the STFT works**: the short-time Fourier transform puts timestamps on a spectrum, and brings the time–frequency resolution trade-off.